# 🔍 Introduction à l'Analyse en Composantes Principales (ACP)
### Projet Lapage — Exploration multi-dimensionnelle du comportement client

**Objectif pédagogique :** Découvrir l'ACP comme outil de visualisation et de synthèse
de plusieurs variables simultanément, en s'appuyant sur les analyses bivariées déjà réalisées.

---
> 📚 Ce notebook s'inspire de la documentation officielle Plotly :  
> https://plotly.com/python/pca-visualization/

## 1. 🤔 Pourquoi l'ACP ? Le problème de la visualisation multi-dimensionnelle

Dans les notebooks précédents, nous avons analysé **4 relations** entre l'âge des clients et :
- 💰 Le montant total des achats
- 📦 La fréquence d'achat
- 🛒 Le panier moyen
- 📚 La catégorie de livres achetés

Ces analyses **bivariées** (1 variable à la fois) sont utiles, mais elles ne montrent pas comment
toutes ces dimensions interagissent **en même temps**.

**L'ACP (Analyse en Composantes Principales)** est une technique qui permet de :
1. **Résumer** plusieurs variables en un petit nombre de composantes
2. **Visualiser** des données à 4, 5, 10 dimensions… en 2D ou 3D
3. **Découvrir** des structures cachées dans les données

> 💡 **Analogie :** Imaginez que vous regardez un objet en 3D depuis plusieurs angles.
> L'ACP choisit l'angle de vue qui montre le **maximum de variation** dans vos données.

## 2. 📦 Chargement des librairies

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("✅ Librairies chargées avec succès")
print(f"  → pandas   {pd.__version__}")
print(f"  → sklearn  (PCA, StandardScaler)")
print(f"  → plotly   (visualisations interactives)")

✅ Librairies chargées avec succès
  → pandas   2.3.3
  → sklearn  (PCA, StandardScaler)
  → plotly   (visualisations interactives)


## 3. 📂 Chargement des données

In [3]:
DATA_PATH = "../data/processed/"

df_source = pd.read_csv(
    DATA_PATH + "transactions_enrichies.csv",
    dtype={
        'id_prod': 'object', 'session_id': 'object', 'client_id': 'object',
        'price': 'float64', 'categ': 'object', 'sex': 'object',
        'birth': 'int64', 'segment_client': 'object', 'age_client': 'int64'
    }
)
df_source['date'] = pd.to_datetime(df_source['date'])

print(f"✅ Données chargées : {len(df_source):,} transactions")
print(f"   Colonnes : {list(df_source.columns)}")
df_source.head(3)

✅ Données chargées : 687,534 transactions
   Colonnes : ['id_prod', 'date', 'session_id', 'client_id', 'price', 'categ', 'sex', 'birth', 'segment_client', 'age_client']


,id_prod,date,session_id,client_id,price,categ,sex,birth,segment_client,age_client
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329,11.99,0,f,1967,B2C,59
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664,19.37,0,m,1960,B2C,66
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580,4.50,0,m,1988,B2C,38


In [10]:
# Retrait des clients B2B
df_source = df_source[df_source['segment_client'] != 'B2B']

## 4. 🏗️ Construction de la matrice client

L'ACP travaille sur un tableau où **chaque ligne = 1 individu** et **chaque colonne = 1 variable**.

Ici, nos individus sont les **clients**. On va construire pour chacun :

| Variable | Description |
|---|---|
| `age_client` | Âge du client |
| `montant_total` | Somme de tous ses achats (€) |
| `nb_achats` | Nombre de transactions (fréquence) |
| `panier_moyen` | Montant moyen par transaction |
| `part_categ_0` | Part des achats en catégorie 0 |
| `part_categ_1` | Part des achats en catégorie 1 |
| `part_categ_2` | Part des achats en catégorie 2 |

> ℹ️ On encode les catégories en **parts (proportions)** pour avoir des valeurs numériques comparables.

In [11]:
# --- Agrégations numériques par client ---
df_clients = df_source.groupby('client_id').agg(
    age_client    = ('age_client', 'first'),
    montant_total = ('price', 'sum'),
    nb_achats     = ('price', 'count'),
    panier_moyen  = ('price', 'mean')
).reset_index()

# --- Encodage des catégories (part de chaque catégorie dans le total des achats) ---
categ_counts = (
    df_source.groupby(['client_id', 'categ'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
categ_counts.columns = ['client_id'] + [f'nb_categ_{c}' for c in categ_counts.columns[1:]]

df_clients = df_clients.merge(categ_counts, on='client_id', how='left')

# Calcul des parts relatives (somme = 1 par client)
categ_cols = [c for c in df_clients.columns if c.startswith('nb_categ_')]
for col in categ_cols:
    part_col = col.replace('nb_categ_', 'part_categ_')
    df_clients[part_col] = df_clients[col] / df_clients['nb_achats']

df_clients = df_clients.drop(columns=categ_cols)

print(f"✅ Matrice client construite : {df_clients.shape[0]:,} clients × {df_clients.shape[1]-1} variables")
print(f"\nVariables disponibles :")
for col in df_clients.columns[1:]:
    print(f"  → {col}")
df_clients.head(3)

✅ Matrice client construite : 8,596 clients × 7 variables

Variables disponibles :
  → age_client
  → montant_total
  → nb_achats
  → panier_moyen
  → part_categ_0
  → part_categ_1
  → part_categ_2


,client_id,age_client,montant_total,nb_achats,panier_moyen,part_categ_0,part_categ_1,part_categ_2
0,c_1,71,629.02,43,14.628372,0.697674,0.279070,0.023256
1,c_10,70,1353.60,58,23.337931,0.344828,0.586207,0.068966
2,c_100,34,254.85,8,31.856250,0.250000,0.625000,0.125000


## 5. 🔗 Rappel : la matrice de corrélation

Avant de lancer l'ACP, visualisons les corrélations entre nos variables.
C'est le point de départ que l'ACP va "synthétiser".

In [12]:
features = ['age_client', 'montant_total', 'nb_achats', 'panier_moyen'] + \
           [c for c in df_clients.columns if c.startswith('part_categ_')]

corr_matrix = df_clients[features].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title="Matrice de corrélation — Variables client",
    width=650, height=550
)
fig.update_layout(font_size=11)
fig.show()

print("\n💡 Interprétation :")
print("   → Les cases bleues foncées = corrélation positive forte")
print("   → Les cases rouges foncées = corrélation négative forte")
print("   → L'ACP va chercher les axes qui capturent au mieux ces structures.")


💡 Interprétation :
   → Les cases bleues foncées = corrélation positive forte
   → Les cases rouges foncées = corrélation négative forte
   → L'ACP va chercher les axes qui capturent au mieux ces structures.


## 6. ⚖️ Standardisation — étape indispensable !

> ⚠️ **Problème :** `montant_total` est en centaines d'euros, `age_client` en dizaines d'années,
> `part_categ` entre 0 et 1. Si on mélange ces échelles telles quelles,
> l'ACP sera **dominée par les grandes valeurs** — ce serait une erreur.

**Solution : la standardisation (z-score)**

Pour chaque variable, on soustrait la moyenne et on divise par l'écart-type :

$$z = \frac{x - \mu}{\sigma}$$

Après standardisation, toutes les variables ont **moyenne = 0** et **écart-type = 1**.

In [13]:
X = df_clients[features].dropna()
client_ids = df_clients.loc[X.index, 'client_id']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Vérification
df_check = pd.DataFrame(X_scaled, columns=features)
print("✅ Après standardisation :")
print(df_check.describe().loc[['mean', 'std']].round(3))
print("\n→ Toutes les moyennes ≈ 0, tous les écarts-types ≈ 1 ✓")

✅ Après standardisation :
      age_client  montant_total  nb_achats  panier_moyen  part_categ_0  \
mean        -0.0           -0.0        0.0          -0.0          -0.0   
std          1.0            1.0        1.0           1.0           1.0   

      part_categ_1  part_categ_2  
mean           0.0           0.0  
std            1.0           1.0  

→ Toutes les moyennes ≈ 0, tous les écarts-types ≈ 1 ✓


## 7. 📊 Combien de composantes garder ?

On commence par faire tourner l'ACP sur **toutes** les composantes, puis on regarde
combien de **variance** chaque composante explique.

Le graphique de la **variance expliquée cumulée** (Scree Plot) aide à décider
combien de composantes sont suffisantes.

> 💡 **Règle empirique :** On garde les composantes qui expliquent au moins **70-80%**
> de la variance totale.

In [14]:
# ACP complète pour analyser la variance
pca_full = PCA()
pca_full.fit(X_scaled)

variance_ratio   = pca_full.explained_variance_ratio_ * 100
variance_cumul   = np.cumsum(variance_ratio)
n_comp           = len(variance_ratio)
composantes      = [f"PC{i+1}" for i in range(n_comp)]

# --- Graphique combiné barres + ligne cumulée ---
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=composantes, y=variance_ratio,
           name="Variance par composante (%)",
           marker_color='steelblue', opacity=0.8),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=composantes, y=variance_cumul, mode='lines+markers',
               name="Variance cumulée (%)",
               line=dict(color='tomato', width=2.5),
               marker=dict(size=8)),
    secondary_y=True
)

# Ligne de seuil 80%
fig.add_hline(y=80, line_dash="dot", line_color="grey",
              annotation_text="Seuil 80%", annotation_position="bottom right",
              secondary_y=True)

fig.update_layout(
    title="Scree Plot — Variance expliquée par composante",
    xaxis_title="Composantes principales",
    legend=dict(x=0.4, y=0.3),
    width=750, height=420
)
fig.update_yaxes(title_text="Variance (%) par composante", secondary_y=False)
fig.update_yaxes(title_text="Variance cumulée (%)", secondary_y=True, range=[0, 105])
fig.show()

print("\n📋 Tableau récapitulatif :")
df_var = pd.DataFrame({
    'Composante': composantes,
    'Variance (%)': variance_ratio.round(1),
    'Cumulée (%)': variance_cumul.round(1)
})
print(df_var.to_string(index=False))


📋 Tableau récapitulatif :
Composante  Variance (%)  Cumulée (%)
       PC1          44.3         44.3
       PC2          30.8         75.1
       PC3          17.1         92.2
       PC4           5.9         98.1
       PC5           1.1         99.2
       PC6           0.8        100.0
       PC7           0.0        100.0


## 8. 🗺️ Visualisation 2D — Les deux premières composantes

On réduit nos données à **2 dimensions** (PC1 et PC2).
Chaque point représente un client.

La couleur encode le **segment client** pour voir si des groupes naturels émergent.

In [15]:
n_comp = 2
pca_2d = PCA(n_components=n_comp)
components_2d = pca_2d.fit_transform(X_scaled)

var_1 = pca_2d.explained_variance_ratio_[0] * 100
var_2 = pca_2d.explained_variance_ratio_[1] * 100

df_plot = pd.DataFrame(components_2d, columns=['PC1', 'PC2'])
df_plot['client_id']      = client_ids.values
df_plot['segment_client'] = df_clients.loc[X.index, 'segment_client'].values  \
                             if 'segment_client' in df_clients.columns \
                             else df_source.groupby('client_id')['segment_client'] \
                                  .first().reindex(client_ids).values
df_plot['age_client']     = df_clients.loc[X.index, 'age_client'].values

fig = px.scatter(
    df_plot, x='PC1', y='PC2',
    color='segment_client',
    hover_data=['client_id', 'age_client'],
    labels={
        'PC1': f'PC1 ({var_1:.1f}% variance)',
        'PC2': f'PC2 ({var_2:.1f}% variance)'
    },
    title=f'ACP 2D — Clients Lapage (variance totale expliquée : {var_1+var_2:.1f}%)',
    opacity=0.6,
    width=800, height=500
)
fig.update_traces(marker=dict(size=5))
fig.show()

print(f"\n💡 PC1 explique {var_1:.1f}% de la variance totale")
print(f"   PC2 explique {var_2:.1f}% de la variance totale")
print(f"   Ensemble : {var_1+var_2:.1f}% — on conserve une bonne partie de l'information")


💡 PC1 explique 44.3% de la variance totale
   PC2 explique 30.8% de la variance totale
   Ensemble : 75.1% — on conserve une bonne partie de l'information


## 9. 🌐 Visualisation 3D — Ajouter une troisième composante

En ajoutant **PC3**, on capture encore plus de variance.
Le graphique 3D est interactif : **cliquez-glissez pour faire pivoter** !

In [16]:
pca_3d = PCA(n_components=3)
components_3d = pca_3d.fit_transform(X_scaled)

total_var_3d = pca_3d.explained_variance_ratio_.sum() * 100

df_3d = pd.DataFrame(components_3d, columns=['PC1', 'PC2', 'PC3'])
df_3d['segment_client'] = df_plot['segment_client'].values
df_3d['age_client']     = df_plot['age_client'].values

fig = px.scatter_3d(
    df_3d, x='PC1', y='PC2', z='PC3',
    color='segment_client',
    hover_data=['age_client'],
    labels={'PC1': 'PC1', 'PC2': 'PC2', 'PC3': 'PC3'},
    title=f'ACP 3D — Variance totale expliquée : {total_var_3d:.1f}%',
    opacity=0.6,
    width=800, height=600
)
fig.update_traces(marker=dict(size=3))
fig.show()

print(f"\n💡 En passant de 2D à 3D, on gagne {total_var_3d-(var_1+var_2):.1f}% de variance supplémentaire")


💡 En passant de 2D à 3D, on gagne 17.1% de variance supplémentaire


## 10. 🧭 Le Biplot — Comprendre ce que représentent PC1 et PC2

Le **biplot** superpose deux informations :
- Les **points** = les clients dans l'espace PC1/PC2
- Les **flèches** = les variables d'origine (loadings)

**Comment lire les flèches :**
- Une flèche **longue** = la variable contribue fortement à ces composantes
- Deux flèches **dans la même direction** = les variables sont corrélées
- Deux flèches **opposées** = les variables sont anticorrélées
- Une flèche **perpendiculaire** à une autre = variables indépendantes

> 💡 C'est ici qu'on fait le lien avec nos **4 analyses bivariées** initiales :
> les variables `age_client`, `montant_total`, `nb_achats`, `panier_moyen`
> vont-elles pointer dans la même direction ?

In [17]:
# Calcul des loadings : eigenvecteurs × racine(valeurs propres)
loadings = pca_2d.components_.T * np.sqrt(pca_2d.explained_variance_)

# Noms courts pour les flèches
feature_labels = {
    'age_client':   'Âge',
    'montant_total':'Montant total',
    'nb_achats':    'Fréquence',
    'panier_moyen': 'Panier moyen',
}
# Pour les catégories
for col in features:
    if col.startswith('part_categ_'):
        feature_labels[col] = col.replace('part_categ_', 'Catég. ')

# --- Scatter des clients ---
fig = px.scatter(
    df_plot, x='PC1', y='PC2',
    color='segment_client',
    opacity=0.4,
    labels={
        'PC1': f'PC1 ({var_1:.1f}%)',
        'PC2': f'PC2 ({var_2:.1f}%)'
    },
    title='Biplot ACP — Clients + Variables',
    width=850, height=560
)
fig.update_traces(marker=dict(size=4))

# --- Flèches des loadings ---
scale = 3.5   # facteur d'échelle visuel
colors_arrow = ['firebrick', 'darkorange', 'forestgreen', 'royalblue',
                'purple', 'brown', 'teal']

for i, feature in enumerate(features):
    lx, ly = loadings[i, 0] * scale, loadings[i, 1] * scale
    label  = feature_labels.get(feature, feature)
    color  = colors_arrow[i % len(colors_arrow)]

    # Flèche
    fig.add_annotation(
        ax=0, ay=0, axref='x', ayref='y',
        x=lx, y=ly,
        showarrow=True, arrowhead=3, arrowsize=1.5,
        arrowwidth=2, arrowcolor=color
    )
    # Label
    fig.add_annotation(
        x=lx * 1.12, y=ly * 1.12,
        text=f"<b>{label}</b>",
        showarrow=False, font=dict(color=color, size=11)
    )

# Axes zéro
fig.add_hline(y=0, line_dash='dot', line_color='lightgrey', line_width=1)
fig.add_vline(x=0, line_dash='dot', line_color='lightgrey', line_width=1)
fig.show()

print("\n📋 Tableau des loadings (contribution de chaque variable à PC1 et PC2) :")
df_loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=features,
    columns=['PC1', 'PC2']
).round(3)
print(df_loadings.to_string())


📋 Tableau des loadings (contribution de chaque variable à PC1 et PC2) :
                 PC1    PC2
age_client    -0.296 -0.405
montant_total -0.092  0.539
nb_achats     -0.329  0.456
panier_moyen   0.523  0.173
part_categ_0  -0.493  0.161
part_categ_1   0.102 -0.480
part_categ_2   0.518  0.227


## 11. 🔲 Scatter Matrix — Vue d'ensemble de toutes les composantes

Pour avoir une vision globale, on peut afficher toutes les paires de composantes
dans une **scatter matrix**. Pratique pour comparer plusieurs PC d'un coup !

In [18]:
pca_all = PCA(n_components=4)
comp_all = pca_all.fit_transform(X_scaled)

labels_dict = {
    str(i): f"PC{i+1} ({pca_all.explained_variance_ratio_[i]*100:.1f}%)"
    for i in range(4)
}

df_matrix = pd.DataFrame(comp_all, columns=[str(i) for i in range(4)])
df_matrix['segment_client'] = df_plot['segment_client'].values

fig = px.scatter_matrix(
    df_matrix,
    dimensions=[str(i) for i in range(4)],
    color='segment_client',
    labels=labels_dict,
    title='Scatter Matrix — 4 premières composantes principales',
    opacity=0.5,
    width=850, height=750
)
fig.update_traces(diagonal_visible=False, marker=dict(size=3))
fig.show()

print("\n💡 Regardez comment PC1 vs PC2 sépare mieux les segments que PC3 vs PC4.")
print("   C'est normal : PC1 et PC2 concentrent le plus de variance informative.")


💡 Regardez comment PC1 vs PC2 sépare mieux les segments que PC3 vs PC4.
   C'est normal : PC1 et PC2 concentrent le plus de variance informative.


## 12. 📝 Synthèse et points clés

### Ce que vous avez appris

| Étape | Outil | Ce qu'on fait |
|---|---|---|
| Préparation | pandas | Construire la matrice client (1 ligne = 1 client) |
| Standardisation | `StandardScaler` | Mettre toutes les variables à la même échelle |
| ACP | `sklearn.PCA` | Calculer les composantes principales |
| Variance expliquée | Scree Plot (plotly) | Choisir le bon nombre de composantes |
| Visualisation 2D/3D | `px.scatter`, `px.scatter_3d` | Explorer les groupes de clients |
| Biplot | `go.Figure` + annotations | Interpréter ce que représentent PC1 et PC2 |
| Scatter matrix | `px.scatter_matrix` | Vue d'ensemble multi-composantes |

---

### Lien avec les 4 analyses précédentes

Les flèches du biplot vous montrent si `age_client`, `montant_total`,
`nb_achats` et `panier_moyen` partagent une **direction commune** (corrélés)
ou s'orientent **différemment** (indépendants).

C'est exactement l'information que vous avez analysée une variable à la fois —
l'ACP vous la montre **en une seule image**.

---

### Pour aller plus loin

- 🔗 [Documentation sklearn PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca)
- 🔗 [Plotly PCA Visualization](https://plotly.com/python/pca-visualization/)
- 📘 Combiner ACP + clustering (K-Means) pour segmenter automatiquement les clients